# Paper Reviewer — Evaluation Benchmarking Plots

Compares three conditions:
- **multi** — 3 agents (A+B+C), 3 iterations
- **baseline** — 1 agent, 1 iteration, no persona
- **nopersona** — 3 agents (NNN), 3 iterations, no persona

In [ ]:
import json
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
EVAL_DIR   = Path("eval_results")          # relative to notebook location
SYSTEMS    = ["our_multi", "our_baseline", "our_nopersona"]
SYS_LABELS = {"our_multi": "Multi (ABC, 3-iter)",
               "our_baseline": "Baseline (1-iter)",
               "our_nopersona": "No-Persona (NNN, 3-iter)"}
COLORS     = {"our_multi": "#4C72B0",
               "our_baseline": "#DD8452",
               "our_nopersona": "#55A868"}
CONF_ORDER = ["ICLR", "ICML", "NeurIPS"]

plt.rcParams.update({"figure.dpi": 120, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
print("Config OK")

In [ ]:
# ── Load latest evaluation JSON ───────────────────────────────────────────────
files = sorted(EVAL_DIR.glob("evaluation_*.json"))
if not files:
    raise FileNotFoundError(f"No evaluation JSON found in {EVAL_DIR}")
eval_path = files[-1]          # use most recent
print(f"Loading: {eval_path}")

with open(eval_path) as f:
    data = json.load(f)

# ── Build per-paper DataFrame ─────────────────────────────────────────────────
rows = []
for paper in data["papers"]:
    base = {
        "paper_id":   paper["paper_id"],
        "conference": paper["conference"],
        "gt_decision": paper["ground_truth"]["accept_or_not"],
        "gt_score":    paper["ground_truth"]["score"],
    }
    for sys in SYSTEMS:
        s = paper["systems"].get(sys)
        if s is None:
            continue
        rows.append({**base, "system": sys,
                     "conference_check": s["conference_check"],
                     "src_strengths":    s["src_strengths"],
                     "src_weaknesses":   s["src_weaknesses"],
                     "src_overall":      s["src_overall"],
                     "norm_score":       s["norm_score"],
                     "norm_gt_score":    s["norm_gt_score"]})

df = pd.DataFrame(rows)
print(f"{len(df)} rows  |  systems: {df['system'].unique().tolist()}")
df.head()

## 1. Aggregate Metrics — Bar Chart

In [ ]:
metrics = [
    ("conference_check", "Conference Check Accuracy"),
    ("src_overall",      "SRC Overall"),
    ("src_strengths",    "SRC Strengths"),
    ("src_weaknesses",   "SRC Weaknesses"),
]

agg = df.groupby("system")[[ m for m,_ in metrics]].mean().reindex(SYSTEMS)

fig, axes = plt.subplots(1, len(metrics), figsize=(14, 4), sharey=False)

for ax, (col, label) in zip(axes, metrics):
    vals  = agg[col].values
    bars  = ax.bar(
        [SYS_LABELS[s] for s in SYSTEMS], vals,
        color=[COLORS[s] for s in SYSTEMS], width=0.55, edgecolor="white"
    )
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    ax.set_title(label, fontsize=11, fontweight="bold")
    ax.set_ylim(0, 1.05)
    ax.set_xticks(range(len(SYSTEMS)))
    ax.set_xticklabels([SYS_LABELS[s] for s in SYSTEMS], rotation=15, ha="right", fontsize=9)
    ax.set_ylabel("Score")

fig.suptitle("Aggregate Metrics by System", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(EVAL_DIR / "plot_aggregate_metrics.png", bbox_inches="tight")
plt.show()

## 2. Per-Paper SRC Overall

In [ ]:
# Pivot so each row = paper, each column = system
pivot = df.pivot_table(index="paper_id", columns="system", values="src_overall")
pivot = pivot.reindex(columns=SYSTEMS)

# Sort papers by conference then name
conf_map  = df.drop_duplicates("paper_id").set_index("paper_id")["conference"].to_dict()
paper_order = sorted(pivot.index, key=lambda p: (conf_map.get(p, ""), p))
pivot = pivot.reindex(paper_order)

x     = np.arange(len(pivot))
width = 0.26

fig, ax = plt.subplots(figsize=(18, 5))
for i, sys in enumerate(SYSTEMS):
    vals = pivot[sys].values.astype(float)
    ax.bar(x + i * width, vals, width, label=SYS_LABELS[sys],
           color=COLORS[sys], alpha=0.85, edgecolor="white")

ax.set_xticks(x + width)
ax.set_xticklabels(pivot.index, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("SRC Overall")
ax.set_title("Per-Paper SRC Overall by System", fontweight="bold")
ax.set_ylim(0, 0.7)
ax.legend(loc="upper right", fontsize=9)

# Conference dividers
prev_conf, prev_i = None, 0
for i, pid in enumerate(pivot.index):
    c = conf_map.get(pid, "")
    if prev_conf and c != prev_conf:
        ax.axvline(i - 0.4, color="gray", lw=0.8, linestyle="--")
    if c != prev_conf:
        ax.text(i + 0.5, 0.66, c, fontsize=8, color="gray", ha="left")
    prev_conf = c

plt.tight_layout()
plt.savefig(EVAL_DIR / "plot_per_paper_src.png", bbox_inches="tight")
plt.show()

## 3. Conference Check Accuracy — by Conference

In [ ]:
conf_acc = (
    df.groupby(["conference", "system"])["conference_check"]
    .mean()
    .unstack("system")
    .reindex(index=CONF_ORDER, columns=SYSTEMS)
)

x     = np.arange(len(CONF_ORDER))
width = 0.26

fig, ax = plt.subplots(figsize=(8, 4))
for i, sys in enumerate(SYSTEMS):
    vals = conf_acc[sys].values.astype(float)
    bars = ax.bar(x + i * width, vals, width, label=SYS_LABELS[sys],
                  color=COLORS[sys], alpha=0.85, edgecolor="white")
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.01,
                    f"{v:.2f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(CONF_ORDER, fontsize=11)
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.15)
ax.set_title("Conference Check Accuracy by Conference", fontweight="bold")
ax.legend(fontsize=9)
ax.axhline(0.5, color="gray", lw=0.8, linestyle=":")

plt.tight_layout()
plt.savefig(EVAL_DIR / "plot_conf_check_by_conference.png", bbox_inches="tight")
plt.show()

## 4. SRC Strengths vs Weaknesses

In [ ]:
agg_sw = df.groupby("system")[["src_strengths", "src_weaknesses"]].mean().reindex(SYSTEMS)

x     = np.arange(len(SYSTEMS))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
bars_s = ax.bar(x - width/2, agg_sw["src_strengths"], width,
                label="SRC Strengths", color="#4C72B0", alpha=0.85, edgecolor="white")
bars_w = ax.bar(x + width/2, agg_sw["src_weaknesses"], width,
                label="SRC Weaknesses", color="#C44E52", alpha=0.85, edgecolor="white")

for bar in list(bars_s) + list(bars_w):
    v = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.005,
            f"{v:.3f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([SYS_LABELS[s] for s in SYSTEMS], rotation=10, ha="right")
ax.set_ylabel("SRC Score")
ax.set_ylim(0, 0.6)
ax.set_title("SRC Strengths vs Weaknesses by System", fontweight="bold")
ax.legend()

plt.tight_layout()
plt.savefig(EVAL_DIR / "plot_src_strengths_vs_weaknesses.png", bbox_inches="tight")
plt.show()

## 5. Score Correlation — Scatter (papers with GT scores)

In [ ]:
from scipy.stats import spearmanr

df_scored = df.dropna(subset=["norm_score", "norm_gt_score"])

fig, axes = plt.subplots(1, len(SYSTEMS), figsize=(13, 4), sharey=True, sharex=True)

for ax, sys in zip(axes, SYSTEMS):
    sub = df_scored[df_scored["system"] == sys]
    if sub.empty:
        ax.set_title(SYS_LABELS[sys])
        ax.text(0.5, 0.5, "No data", ha="center", transform=ax.transAxes)
        continue

    conf_colors = {"ICLR": "#4C72B0", "ICML": "#DD8452", "NeurIPS": "#55A868"}
    for conf, grp in sub.groupby("conference"):
        ax.scatter(grp["norm_gt_score"], grp["norm_score"],
                   c=conf_colors.get(conf, "gray"), label=conf,
                   s=60, alpha=0.8, edgecolors="white", linewidths=0.5)

    rho, pval = spearmanr(sub["norm_gt_score"], sub["norm_score"])
    ax.set_title(f"{SYS_LABELS[sys]}\nρ={rho:.3f}  p={pval:.3f}", fontsize=9, fontweight="bold")
    ax.set_xlabel("Normalised GT Score")
    ax.plot([0, 1], [0, 1], "--", color="gray", lw=0.8, alpha=0.5)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)

axes[0].set_ylabel("Normalised Our Score")
handles = [mpatches.Patch(color=c, label=k) for k,c in conf_colors.items()]
axes[-1].legend(handles=handles, fontsize=8, loc="lower right")

fig.suptitle("Score Correlation: Our Normalised Score vs GT Normalised Rating",
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(EVAL_DIR / "plot_score_correlation.png", bbox_inches="tight")
plt.show()

## 6. Accept / Reject Decision Breakdown

In [ ]:
fig, axes = plt.subplots(1, len(SYSTEMS), figsize=(12, 4))

for ax, sys in zip(axes, SYSTEMS):
    sub = df[df["system"] == sys].copy()
    sub["correct"] = sub["conference_check"].map({True: "Correct", False: "Wrong", None: "N/A"})

    # Count correct / wrong per gt_decision
    ct = sub.groupby(["gt_decision", "correct"]).size().unstack(fill_value=0)
    for col in ["Correct", "Wrong"]:
        if col not in ct.columns:
            ct[col] = 0

    labels = ct.index.tolist()
    correct = ct["Correct"].values
    wrong   = ct["Wrong"].values
    x = np.arange(len(labels))

    ax.bar(x, correct, label="Correct", color="#55A868", edgecolor="white", alpha=0.85)
    ax.bar(x, wrong,   bottom=correct, label="Wrong", color="#C44E52", edgecolor="white", alpha=0.85)

    for i, (c, w) in enumerate(zip(correct, wrong)):
        total = c + w
        if total > 0:
            ax.text(i, total + 0.1, f"{c}/{total}", ha="center", fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels([l.capitalize() for l in labels])
    ax.set_title(SYS_LABELS[sys], fontsize=9, fontweight="bold")
    ax.set_ylabel("# Papers")
    if ax == axes[0]:
        ax.legend(fontsize=8)

fig.suptitle("Accept/Reject Decision Breakdown (Correct vs Wrong)",
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(EVAL_DIR / "plot_decision_breakdown.png", bbox_inches="tight")
plt.show()

## 7. Radar Chart — All Metrics at a Glance

In [ ]:
radar_metrics = [
    ("conference_check", "Conf. Check\nAccuracy"),
    ("src_overall",      "SRC Overall"),
    ("src_strengths",    "SRC\nStrengths"),
    ("src_weaknesses",   "SRC\nWeaknesses"),
]

agg_radar = df.groupby("system")[[m for m,_ in radar_metrics]].mean().reindex(SYSTEMS)

N      = len(radar_metrics)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]   # close the loop

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

for sys in SYSTEMS:
    vals = agg_radar.loc[sys].values.tolist()
    vals += vals[:1]
    ax.plot(angles, vals, color=COLORS[sys], linewidth=2, label=SYS_LABELS[sys])
    ax.fill(angles, vals, color=COLORS[sys], alpha=0.12)

ax.set_xticks(angles[:-1])
ax.set_xticklabels([lbl for _, lbl in radar_metrics], fontsize=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8])
ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8"], fontsize=7, color="gray")
ax.set_title("System Comparison — Radar Chart", fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=9)

plt.tight_layout()
plt.savefig(EVAL_DIR / "plot_radar.png", bbox_inches="tight")
plt.show()

## 8. Summary Table

In [ ]:
from scipy.stats import spearmanr

summary_rows = []
for sys in SYSTEMS:
    sub = df[df["system"] == sys]
    scored = sub.dropna(subset=["norm_score", "norm_gt_score"])
    rho = np.nan
    if len(scored) >= 3:
        rho, _ = spearmanr(scored["norm_gt_score"], scored["norm_score"])
    summary_rows.append({
        "System":            SYS_LABELS[sys],
        "n papers":          len(sub["paper_id"].unique()),
        "Conf. Check Acc":   f"{sub['conference_check'].mean():.3f}",
        "Score Spearman ρ":  f"{rho:.3f}" if not np.isnan(rho) else "N/A",
        "SRC Strengths":     f"{sub['src_strengths'].mean():.3f}",
        "SRC Weaknesses":    f"{sub['src_weaknesses'].mean():.3f}",
        "SRC Overall":       f"{sub['src_overall'].mean():.3f}",
    })

summary_df = pd.DataFrame(summary_rows).set_index("System")
print(summary_df.to_string())
summary_df